### Convert MRI reference

In [20]:
# Notebook to compare visualisation results for NIfTI vs OME-zarr using Neuroglancer & nifti-zarr-py
import subprocess
from pathlib import Path

inp_file = Path('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409/dti_FA_in_PSOCT.nii.gz')
out_folder = Path('/Users/Vasilis/Downloads/CMC_results/Zarr_ngtools_test')

subprocess.run(["nii2zarr",
                inp_file,
                out_folder / inp_file.name.replace('.nii.gz', '.zarr'),
                # "--ome-version", str(0.5)
                ])


CompletedProcess(args=['nii2zarr', PosixPath('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409/dti_FA_in_PSOCT.nii.gz'), PosixPath('/Users/Vasilis/Downloads/CMC_results/Zarr_ngtools_test/dti_FA_in_PSOCT.zarr')], returncode=0)

In [22]:
# Notebook to compare visualisation results for NIfTI vs OME-zarr using Neuroglancer & nifti-zarr-py
import subprocess
from pathlib import Path

inp_file = Path('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409/Retardance/Slice_101_EnR_hdr.nii.gz')
out_folder = Path('/Users/Vasilis/Downloads/CMC_results/Zarr_ngtools_test')

subprocess.run(["nii2zarr",
                inp_file,
                out_folder / inp_file.name.replace('.nii.gz', '.zarr'),
                # "--ome-version", str(0.5)
                ])


CompletedProcess(args=['nii2zarr', PosixPath('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409/Retardance/Slice_101_EnR_hdr.nii.gz'), PosixPath('/Users/Vasilis/Downloads/CMC_results/Zarr_ngtools_test/Slice_101_EnR_hdr.zarr')], returncode=0)

### Create a 3D Zarr from 2D slices

In [ ]:
# ### Resample 2D slices to reference space

# from pathlib import Path
# from fsl.data.image import Image
# from fsl.utils.image.resample import resampleToReference
# import numpy as np

# ref_file = Path('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409/dti_FA_in_PSOCT.nii.gz')
# slice_dir = Path('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409/Retardance/lowres')
# output_dir = Path('/Users/Vasilis/Downloads/CMC_results/Zarr_ngtools_test/resliced_new')

# output_dir.mkdir(exist_ok=True)

# # Load reference
# ref_img = Image(str(ref_file))

# # Resample each slice to reference space
# slice_files = sorted(slice_dir.glob('*.nii.gz'))
# for slice_file in slice_files:
#     output_file = output_dir / slice_file.name.replace('.nii.gz', '_resampled.nii.gz')
    
#     # Load slice
#     slice_img = Image(str(slice_file))
    
#     # Pad 2D slice to 3D by repeating along first axis
#     # slice_img.shape is 2D, we need to make it 3D
#     slice_data = slice_img.data.squeeze()
#     if slice_data.ndim == 2:
#         # Pad to 3D: repeat the 2D data along x-axis to match reference depth
#         ref_x = ref_img.shape[0]
#         padded_3d = np.repeat(slice_data[np.newaxis, :, :], ref_x, axis=0)
#         # Create 3D image with REFERENCE affine so data positions at index 0
#         # This ensures the padded slice aligns properly with reference space
#         slice_3d = Image(padded_3d.astype(np.float32), header=ref_img.header)
#     else:
#         slice_3d = slice_img

#     # Now resample the 3D padded slice to reference space
#     resampled_data = resampleToReference(slice_3d, ref_img)[0]
    
#     # Wrap resampled data in an Image with reference header
#     resampled_img = Image(resampled_data.astype(np.float32), header=ref_img.header)
    
#     # Save using FSL Image save method
#     resampled_img.save(str(output_file))


In [ ]:
from pathlib import Path
import zarr
import numpy as np
import nibabel as nib
from fsl.data.image import Image
from fsl.utils.image.resample import resampleToReference

resample_flag = True

out_folder = Path('/Users/Vasilis/Downloads/CMC_results/Zarr_ngtools_test')
# slice_dir = Path('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409/Retardance/lowres')
slice_dir = Path('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409/Retardance')
slice_files = sorted(slice_dir.glob('*.nii.gz'))

if len(slice_files) == 0:
    raise ValueError(f'No slices found in {slice_dir}')

# Load reference image (once)
ref_file = Path('/Users/Vasilis/Downloads/CMC_results/Vlad_fnirt_Retardance_20260409/dti_FA_in_PSOCT.nii.gz')
ref_img_fsl = Image(str(ref_file))
ref_img_nib = nib.load(str(ref_file))
ref_affine = ref_img_nib.affine.copy()

# Get reference dimensions
ref_x = ref_img_fsl.shape[0]
ref_y = ref_img_fsl.shape[1]
ref_z = ref_img_fsl.shape[2]

# Create output Zarr group
out_folder.mkdir(parents=True, exist_ok=True)
# out_name = out_folder / 'vol_3D_lowres_resampled.zarr'
out_name = out_folder / 'vol_3D_resampled.zarr'
root = zarr.open_group(str(out_name), mode='w', zarr_format=2)

# ==== Determine Zarr dimensions ====
if resample_flag:
    # Resample first 2D slice to get dimensions
    first_slice_img = Image(str(slice_files[0]))
    first_slice_data = first_slice_img.data.squeeze()
    padded_3d = np.repeat(first_slice_data[np.newaxis, :, :], ref_x, axis=0)
    first_padded = Image(padded_3d.astype(np.float32), header=ref_img_nib.header)
    first_resampled = resampleToReference(first_padded, ref_img_fsl)[0]
    y, z = first_resampled.shape[1], first_resampled.shape[2]
else:
    # Get dimensions from first pre-resampled 3D file
    first_img = nib.load(str(slice_files[0]))
    first_plane = np.asarray(first_img.dataobj, dtype=np.float32).squeeze()
    y, z = first_plane.shape

x = len(slice_files)

# Store as (Z, Y, X) to match axis annotation [z, y, x]
arr0 = root.create_array(
    '0',
    shape=(z, y, x),
    chunks=(1, min(1024, y), min(1024, x)),
    dtype='float32',
)

# Extract voxel size and origin
voxel_x = float(np.linalg.norm(ref_affine[:3, 0]))
voxel_y = float(np.linalg.norm(ref_affine[:3, 1]))
voxel_z = float(np.linalg.norm(ref_affine[:3, 2]))
origin = ref_affine[:3, 3].copy()

# Build diagonal affine
affine_xyz = np.array([
    [voxel_x, 0.0, 0.0, float(origin[0])],
    [0.0, voxel_y, 0.0, float(origin[1])],
    [0.0, 0.0, voxel_z, float(origin[2])],
], dtype=np.float64)

# Orientation metadata
if Path(out_folder / 'dti_FA_in_PSOCT.zarr').exists():
    ref_root = zarr.open_group(str(out_folder / 'dti_FA_in_PSOCT.zarr'), mode='r')
    orientation = ref_root['nifti'].attrs.get('Orientation', {'x': 'l', 'y': 'a', 'z': 's'})
else:
    orientation = {'x': 'l', 'y': 'a', 'z': 's'}

# Adaptive multiscale schedule
max_levels = 6
xy_switch_threshold = 512
min_yz_size = 32

levels = []
cum_fx, cum_fy, cum_fz = 1, 1, 1
cur_x, cur_y, cur_z = x, y, z

for level_idx in range(1, max_levels + 1):
    if min(cur_y, cur_z) < min_yz_size:
        break

    use_x_downsample = (min(cur_y, cur_z) <= xy_switch_threshold) and (cur_x >= 2)
    step_fx = 2 if use_x_downsample else 1
    step_fy = 2 if cur_y >= 2 else 1
    step_fz = 2 if cur_z >= 2 else 1

    if (step_fx, step_fy, step_fz) == (1, 1, 1):
        break

    cum_fx *= step_fx
    cum_fy *= step_fy
    cum_fz *= step_fz

    out_x = x // cum_fx
    out_y = y // cum_fy
    out_z = z // cum_fz
    if out_x == 0 or out_y == 0 or out_z == 0:
        break

    level_name = str(level_idx)
    level_array = root.create_array(
        level_name,
        shape=(out_z, out_y, out_x),
        chunks=(1, min(512, out_y), min(512, out_x)),
        dtype='float32',
    )

    levels.append({
        'name': level_name,
        'array': level_array,
        'cum_fx': cum_fx,
        'cum_fy': cum_fy,
        'cum_fz': cum_fz,
        'out_x': out_x,
        'out_y': out_y,
        'out_z': out_z,
        'x_trim': out_x * cum_fx,
        'y_trim': out_y * cum_fy,
        'z_trim': out_z * cum_fz,
        'buffer': np.zeros((out_z, out_y), dtype=np.float32),
        'buffer_count': 0,
        'x_out_idx': 0,
    })

    cur_x, cur_y, cur_z = out_x, out_y, out_z

# ==== Main loop: resample + stack or just stack ====
overall_min = float('inf')
overall_max = float('-inf')

if resample_flag:
    # Mode 1: Resample each 2D slice to reference space, then stack
    for i, slice_file in enumerate(slice_files):
        # Load 2D slice
        slice_img = Image(str(slice_file))
        slice_data = slice_img.data.squeeze()
        
        if slice_data.ndim == 2:
            # Pad to 3D with reference affine
            padded_3d = np.repeat(slice_data[np.newaxis, :, :], ref_x, axis=0)
            slice_3d = Image(padded_3d.astype(np.float32), header=ref_img_nib.header)
        else:
            slice_3d = slice_img
        
        # Resample to reference space
        resampled_data = resampleToReference(slice_3d, ref_img_fsl)[0]
        
        # Extract plane [0, :, :] (first X index)
        data_yz = resampled_data[0, :, :].astype(np.float32)
        
        if data_yz.shape != (y, z):
            raise ValueError(f'shape mismatch for {slice_file.name}: {data_yz.shape} != {(y, z)}')

        # Store directly to Zarr: data is (Y, Z), store transposed to arr0[Z, Y, X]
        arr0[:, :, i] = data_yz.T

        # Track min/max
        smin = float(data_yz.min())
        smax = float(data_yz.max())
        if smin < overall_min:
            overall_min = smin
        if smax > overall_max:
            overall_max = smax

        # Write pyramid levels from the same resampled slice
        for lvl in levels:
            if i >= lvl['x_trim']:
                continue

            d2 = data_yz[:lvl['y_trim'], :lvl['z_trim']]
            yz_down = d2.reshape(
                lvl['out_y'], lvl['cum_fy'], lvl['out_z'], lvl['cum_fz']
            ).mean(axis=(1, 3), dtype=np.float32)

            if lvl['cum_fx'] == 1:
                lvl['array'][:, :, i] = yz_down.T
            else:
                lvl['buffer'] += yz_down.T
                lvl['buffer_count'] += 1
                if lvl['buffer_count'] == lvl['cum_fx']:
                    lvl['array'][:, :, lvl['x_out_idx']] = (lvl['buffer'] / lvl['cum_fx']).astype(np.float32, copy=False)
                    lvl['x_out_idx'] += 1
                    lvl['buffer'].fill(0)
                    lvl['buffer_count'] = 0
        
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/{len(slice_files)} slices")

else:
    # Mode 2: Load pre-resampled 3D volumes and extract planes
    for i, p in enumerate(slice_files):
        # Lazy load only the plane we need: [0, :, :] from the 3D volume
        img = nib.load(str(p))
        data_yz = np.asarray(img.dataobj, dtype=np.float32).squeeze()
        
        if data_yz.shape != (y, z):
            raise ValueError(f'shape mismatch for {p.name}: {data_yz.shape} != {(y, z)}')

        # Store directly to Zarr: data is (Y, Z), store transposed to arr0[Z, Y, X]
        arr0[:, :, i] = data_yz.T

        # Track min/max
        smin = float(data_yz.min())
        smax = float(data_yz.max())
        if smin < overall_min:
            overall_min = smin
        if smax > overall_max:
            overall_max = smax

        # Write pyramid levels from the same slice
        for lvl in levels:
            if i >= lvl['x_trim']:
                continue

            d2 = data_yz[:lvl['y_trim'], :lvl['z_trim']]
            yz_down = d2.reshape(
                lvl['out_y'], lvl['cum_fy'], lvl['out_z'], lvl['cum_fz']
            ).mean(axis=(1, 3), dtype=np.float32)

            if lvl['cum_fx'] == 1:
                lvl['array'][:, :, i] = yz_down.T
            else:
                lvl['buffer'] += yz_down.T
                lvl['buffer_count'] += 1
                if lvl['buffer_count'] == lvl['cum_fx']:
                    lvl['array'][:, :, lvl['x_out_idx']] = (lvl['buffer'] / lvl['cum_fx']).astype(np.float32, copy=False)
                    lvl['x_out_idx'] += 1
                    lvl['buffer'].fill(0)
                    lvl['buffer_count'] = 0
        
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/{len(slice_files)} files")

# Metadata
root.attrs['data_min'] = float(overall_min)
root.attrs['data_max'] = float(overall_max)

datasets = []
for idx in range(0, len(levels) + 1):
    if idx == 0:
        fx = fy = fz = 1
        path = '0'
    else:
        lvl = levels[idx - 1]
        fx, fy, fz = lvl['cum_fx'], lvl['cum_fy'], lvl['cum_fz']
        path = lvl['name']

    sx = voxel_x * fx
    sy = voxel_y * fy
    sz = voxel_z * fz

    tx = 0.5 * (fx - 1) * voxel_x
    ty = 0.5 * (fy - 1) * voxel_y
    tz = 0.5 * (fz - 1) * voxel_z

    datasets.append({
        'path': path,
        'coordinateTransformations': [
            {'type': 'scale', 'scale': [sz, sy, sx]},
            {'type': 'translation', 'translation': [tz, ty, tx]},
        ],
    })

root.attrs['multiscales'] = [{
    'axes': [
        {'name': 'z', 'type': 'space', 'unit': 'millimeter'},
        {'name': 'y', 'type': 'space', 'unit': 'millimeter'},
        {'name': 'x', 'type': 'space', 'unit': 'millimeter'},
    ],
    'coordinateTransformations': [
        {'type': 'scale', 'scale': [1.0, 1.0, 1.0]}
    ],
    'datasets': datasets,
    'name': '',
    'type': 'median window 2x2x2',
    'version': '0.4',
}]

# Store nifti metadata
nifti_group = root.create_group('nifti')
nifti_group.create_array('0', shape=(348,), chunks=(348,), dtype='uint8')

nifti_group.attrs.update({
    'Affine': affine_xyz[:3, :4].tolist(),
    'Dim': [int(x), int(y), int(z)],
    'VoxelSize': [voxel_x, voxel_y, voxel_z],
    'Orientation': orientation,
    'SForm': 'aligned_anat',
    'QForm': '',
    'QuaternOffset': {
        'x': float(origin[0]),
        'y': float(origin[1]),
        'z': float(origin[2]),
    },
    'Unit': {'L': 'mm', 'T': 's'},
    'DataType': 'single',
    'BitDepth': 32,
    'NIIHeaderSize': 348,
})

print('Done. Wrote', out_name)
print('scale0 shape:', arr0.shape)
print('levels:', [lvl['name'] for lvl in levels])
print('min/max:', root.attrs['data_min'], root.attrs['data_max'])

Processed 50/245 slices
Processed 100/245 slices
Processed 150/245 slices
Processed 200/245 slices
Done. Wrote /Users/Vasilis/Downloads/CMC_results/Zarr_ngtools_test/vol_3D_lowres_resampled.zarr
scale0 shape: (820, 1270, 245)
levels: ['1', '2', '3', '4', '5']
min/max: 0.0 1.5615602731704712


In [ ]:
# Run: ngtools
# For each file, use: load notebooks/Moe_cc_Ret_Yael/PSOCT_cc_Ret_in_DTI.zarr